# Final Multimodal Evaluation



In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f'IN_COLAB={IN_COLAB}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')

Mounted at /content/drive
IN_COLAB=True
PROJECT_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json
import re

import numpy as np
import pandas as pd
import torch

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

TUNING_DIR = PROJECT_ROOT / 'experiments' / '5_residual_hyper'
FINAL_DIR = PROJECT_ROOT / 'experiments' / '6_final_multimodal'
RESULTS_DIR = FINAL_DIR / 'results'
SUMMARY_DIR = FINAL_DIR / 'summary'

BASELINE_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
BASELINE_RESULTS_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'

for p in [FINAL_DIR, RESULTS_DIR, SUMMARY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ENCODERS = {
    'resnet18': {'config_id': 'cfg_23', 'img_feat_dim': 512, 'img_proj_dim': 4, 'img_dropout': 0.0, 'lr': 1e-4, 'weight_decay': 0.0},
    'vit':      {'config_id': 'cfg_17', 'img_feat_dim': 768, 'img_proj_dim': 4, 'img_dropout': 0.0, 'lr': 3e-5, 'weight_decay': 0.0},
    'clip':     {'config_id': 'cfg_09', 'img_feat_dim': 768, 'img_proj_dim': 4, 'img_dropout': 0.0, 'lr': 1e-4, 'weight_decay': 1e-4},
    'dinov2':   {'config_id': 'cfg_34', 'img_feat_dim': 768, 'img_proj_dim': 4, 'img_dropout': 0.3, 'lr': 1e-3, 'weight_decay': 1e-5},
}

print('Final evaluation directory:', FINAL_DIR)
display(pd.DataFrame(ENCODERS).T)

Final evaluation directory: /content/drive/MyDrive/dissertation/experiments/6_final_multimodal


,config_id,img_feat_dim,img_proj_dim,img_dropout,lr,weight_decay
resnet18,cfg_23,512,4,0.0,0.0001,0.0
vit,cfg_17,768,4,0.0,0.00003,0.0
clip,cfg_09,768,4,0.0,0.0001,0.0001
dinov2,cfg_34,768,4,0.3,0.001,0.00001


## 1. Verify the locked validation selections

In [3]:
for encoder, spec in ENCODERS.items():
    best_path = TUNING_DIR / 'results' / encoder / 'summary' / f'{encoder}_best_config.json'
    if not best_path.exists():
        raise FileNotFoundError(f'Missing locked validation selection: {best_path}')

    with open(best_path) as f:
        best = json.load(f)

    selected = best['selected']
    checks = {
        'config_id': selected['config_id'] == spec['config_id'],
        'img_proj_dim': int(selected['img_proj_dim']) == spec['img_proj_dim'],
        'img_dropout': np.isclose(float(selected['img_dropout']), spec['img_dropout']),
        'lr': np.isclose(float(selected['lr']), spec['lr']),
        'weight_decay': np.isclose(float(selected['weight_decay']), spec['weight_decay']),
    }

    if not all(checks.values()):
        raise RuntimeError(f'{encoder}: locked configuration does not match tuning result: {checks}')

    print(f'{encoder:>8}: {spec["config_id"]} verified')

resnet18: cfg_23 verified
     vit: cfg_17 verified
    clip: cfg_09 verified
  dinov2: cfg_34 verified


## 2. Check that every selected seed checkpoint exists

In [4]:
def find_best_checkpoint(checkpoint_dir):
    if not checkpoint_dir.exists():
        raise FileNotFoundError(f'Checkpoint directory not found: {checkpoint_dir}')

    files = [p for p in checkpoint_dir.rglob('*') if p.is_file() and p.suffix.lower() in {'.pt', '.pth', '.ckpt'}]
    if not files:
        raise FileNotFoundError(f'No PyTorch checkpoint found in: {checkpoint_dir}')

    best_named = [p for p in files if 'best' in p.name.lower()]
    candidates = best_named if best_named else files

    if len(candidates) != 1:
        names = '\n'.join(str(p) for p in candidates)
        raise RuntimeError(f'Expected one best checkpoint in {checkpoint_dir}, found {len(candidates)}:\n{names}')

    return candidates[0]


checkpoint_map = {}

for encoder, spec in ENCODERS.items():
    checkpoint_map[encoder] = {}

    for seed in SEEDS:
        checkpoint_dir = TUNING_DIR / 'checkpoints' / encoder / spec['config_id'] / f'seed_{seed}'
        checkpoint_map[encoder][seed] = find_best_checkpoint(checkpoint_dir)

    print(f'{encoder:>8}: {len(checkpoint_map[encoder])}/10 selected checkpoints found')

print('\nCheckpoint audit passed. Test data have not been loaded.')

resnet18: 10/10 selected checkpoints found
     vit: 10/10 selected checkpoints found
    clip: 10/10 selected checkpoints found
  dinov2: 10/10 selected checkpoints found

Checkpoint audit passed. Test data have not been loaded.


## 3. Load a trained residual checkpoint safely

In [5]:
def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        return checkpoint['model_state_dict']
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        return checkpoint['state_dict']
    if isinstance(checkpoint, dict) and checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
        return checkpoint
    raise RuntimeError('Unsupported checkpoint format. Inspect the checkpoint before evaluating the test set.')


def load_selected_model(encoder, seed, device):
    spec = ENCODERS[encoder]
    base_model = load_frozen_baseline(seed, BASELINE_CHECKPOINT_DIR)

    model = FrozenImageOnlyResidualPredictor(
        base_model=base_model, img_feat_dim=spec['img_feat_dim'], img_proj_dim=spec['img_proj_dim'], img_dropout=spec['img_dropout'])

    checkpoint = torch.load(checkpoint_map[encoder][seed], map_location=device, weights_only=False)
    state_dict = extract_state_dict(checkpoint)

    # Handle checkpoints saved from DataParallel.
    if state_dict and all(k.startswith('module.') for k in state_dict):
        state_dict = {k[len('module.'):]: v for k, v in state_dict.items()}

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()
    return model

## 4. Dry-run checkpoint loading before opening the test split

In [6]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

for encoder in ENCODERS:
    for seed in SEEDS:
        model = load_selected_model(encoder, seed, DEVICE)
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    print(f'{encoder:>8}: 10/10 checkpoints loaded successfully')

print('\nAll 40 selected checkpoints passed strict loading.')

Device: cuda
resnet18: 10/10 checkpoints loaded successfully
     vit: 10/10 checkpoints loaded successfully
    clip: 10/10 checkpoints loaded successfully
  dinov2: 10/10 checkpoints loaded successfully

All 40 selected checkpoints passed strict loading.


## 5. Load the held-out test split

In [7]:
stats = data.load_cgm_stats()
test_participant_ids = data.load_multimodal_participant_ids('test')
TEST_PARTICIPANT_IDS = sorted(set(test_participant_ids))

test_loaders = {}

for encoder, spec in ENCODERS.items():
    loader = data.load_multimodal_tensors('test', encoder_name=encoder, shuffle=False)
    if len(loader.dataset) != len(test_participant_ids):
        raise ValueError(f'{encoder}: test participant IDs do not align with test samples.')

    sample_batch = next(iter(loader))
    img_feat = sample_batch[3]

    if img_feat.shape[-1] != spec['img_feat_dim']:
        raise ValueError(f'{encoder}: expected feature dimension {spec["img_feat_dim"]}, got {img_feat.shape[-1]}.')

    test_loaders[encoder] = loader
    print(f'{encoder:>8}: n={len(loader.dataset)}, image feature dim={img_feat.shape[-1]}')

print('Test participants:', TEST_PARTICIPANT_IDS)

resnet18: n=16818, image feature dim=512
     vit: n=16818, image feature dim=768
    clip: n=16818, image feature dim=768
  dinov2: n=16818, image feature dim=768
Test participants: [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [8]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device))
    return pred, target.to(device)


def h_entry(d, h):
    if str(h) in d:
        return d[str(h)]
    if h in d:
        return d[h]
    raise KeyError(f'Horizon {h} not found. Keys={list(d.keys())}')


def p_entry(d, pid):
    if str(pid) in d:
        return d[str(pid)]
    if pid in d:
        return d[pid]
    raise KeyError(f'Participant {pid} not found. Keys={list(d.keys())}')

## 6. Final held-out test evaluation

In [9]:
for encoder, spec in ENCODERS.items():
    encoder_dir = RESULTS_DIR / encoder
    encoder_dir.mkdir(parents=True, exist_ok=True)

    print('\n' + '=' * 90)
    print(f'FINAL TEST: {encoder} | {spec["config_id"]}')
    print('=' * 90)

    for seed in SEEDS:
        save_path = encoder_dir / f'{encoder}_{spec["config_id"]}_seed{seed}_test.json'

        if save_path.exists():
            print(f'  seed={seed}: already evaluated')
            continue

        model = load_selected_model(encoder, seed, DEVICE)
        y_pred_test, y_true_test = train.predict(model, test_loaders[encoder], forward_fn=forward_fn)

        test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
        per_participant_test = metrics.evaluate_per_participant(
            y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

        for pid in TEST_PARTICIPANT_IDS:
            _ = p_entry(per_participant_test, pid)

        payload = {
            'encoder': encoder,
            'config_id': spec['config_id'],
            'seed': seed,
            'hyperparameters': spec,
            'checkpoint': str(checkpoint_map[encoder][seed]),
            'selection_data': 'validation only',
            'evaluation_split': 'held-out test',
            'test_results': test_results,
            'test_results_per_participant': per_participant_test,
        }

        with open(save_path, 'w') as f:
            json.dump(payload, f, indent=2)

        print(f'  seed={seed}: saved')

        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print('\nFinal multimodal test evaluation complete.')


FINAL TEST: resnet18 | cfg_23
  seed=7: saved
  seed=14: saved
  seed=21: saved
  seed=28: saved
  seed=35: saved
  seed=42: saved
  seed=49: saved
  seed=56: saved
  seed=63: saved
  seed=70: saved

FINAL TEST: vit | cfg_17
  seed=7: saved
  seed=14: saved
  seed=21: saved
  seed=28: saved
  seed=35: saved
  seed=42: saved
  seed=49: saved
  seed=56: saved
  seed=63: saved
  seed=70: saved

FINAL TEST: clip | cfg_09
  seed=7: saved
  seed=14: saved
  seed=21: saved
  seed=28: saved
  seed=35: saved
  seed=42: saved
  seed=49: saved
  seed=56: saved
  seed=63: saved
  seed=70: saved

FINAL TEST: dinov2 | cfg_34
  seed=7: saved
  seed=14: saved
  seed=21: saved
  seed=28: saved
  seed=35: saved
  seed=42: saved
  seed=49: saved
  seed=56: saved
  seed=63: saved
  seed=70: saved

Final multimodal test evaluation complete.


## 7. Load the seed-matched CGM+Time test reference

In [14]:
CGM_RESULTS_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'results' / 'cgm_only'
CGM_FILENAME = 'cgm_only_seed{seed}_results.json'

CGM_TIME_RESULTS_DIR = (
    PROJECT_ROOT / 'experiments' / '3_initial_multimodal'
    / 'multimodal' / 'results' / 'gated_cgm_time'
)
CGM_TIME_FILENAME = 'gated_cgm_time_seed{seed}_results.json'

MODEL_NAMES = {
    'cgm': 'CGM',
    'cgm_time': 'CGM+Time',
    'resnet18': '+ ResNet18',
    'vit': '+ ViT-B/16',
    'clip': '+ CLIP ViT-B/16',
    'dinov2': '+ DINOv2 ViT-B/14',
}

all_results = {model: {} for model in MODEL_NAMES}

for seed in SEEDS:
    cgm_path = CGM_RESULTS_DIR / CGM_FILENAME.format(seed=seed)
    cgm_time_path = CGM_TIME_RESULTS_DIR / CGM_TIME_FILENAME.format(seed=seed)

    if not cgm_path.exists():
        raise FileNotFoundError(
            f'CGM result not found: {cgm_path}\n'
            'Update CGM_RESULTS_DIR / CGM_FILENAME to your existing CGM-only JSON files.')
    if not cgm_time_path.exists():
        raise FileNotFoundError(f'CGM+Time result not found: {cgm_time_path}')

    with open(cgm_path) as f:
        all_results['cgm'][seed] = json.load(f)
    with open(cgm_time_path) as f:
        all_results['cgm_time'][seed] = json.load(f)

    for encoder, spec in ENCODERS.items():
        path = RESULTS_DIR / encoder / f'{encoder}_{spec["config_id"]}_seed{seed}_test.json'
        with open(path) as f:
            all_results[encoder][seed] = json.load(f)

for model in MODEL_NAMES:
    print(f'{MODEL_NAMES[model]:<22} {len(all_results[model])}/10 seeds')

CGM                    10/10 seeds
CGM+Time               10/10 seeds
+ ResNet18             10/10 seeds
+ ViT-B/16             10/10 seeds
+ CLIP ViT-B/16        10/10 seeds
+ DINOv2 ViT-B/14      10/10 seeds


In [15]:
NUMERIC_METRICS = ['rmse', 'mae', 'grmse']
numeric_rows = []

for model, seed_dict in all_results.items():
    row = {'model_key': model, 'model': MODEL_NAMES[model]}

    for h in HORIZONS:
        for metric in NUMERIC_METRICS:
            values = np.asarray([
                float(h_entry(seed_dict[seed]['test_results'], h)[metric]) for seed in SEEDS])

            row[f'{metric}_{h}'] = float(np.mean(values))
            row[f'{metric}_sd1_{h}'] = float(np.std(values, ddof=1))

            if all('test_results_per_participant' in seed_dict[seed] for seed in SEEDS):
                participant_means = []
                for pid in TEST_PARTICIPANT_IDS:
                    pid_values = [
                        float(h_entry(p_entry(seed_dict[seed]['test_results_per_participant'], pid), h)[metric])
                        for seed in SEEDS
                    ]
                    participant_means.append(np.mean(pid_values))
                row[f'{metric}_sd2_{h}'] = float(np.std(participant_means, ddof=1))
            else:
                row[f'{metric}_sd2_{h}'] = np.nan

    numeric_rows.append(row)

numeric_summary = pd.DataFrame(numeric_rows)
display(numeric_summary.round(4))

,model_key,model,rmse_15,rmse_sd1_15,rmse_sd2_15,mae_15,mae_sd1_15,mae_sd2_15,grmse_15,grmse_sd1_15,...,grmse_sd2_90,rmse_120,rmse_sd1_120,rmse_sd2_120,mae_120,mae_sd1_120,mae_sd2_120,grmse_120,grmse_sd1_120,grmse_sd2_120
0,cgm,CGM,11.2229,0.0853,1.8889,7.5993,0.0905,1.4596,12.4160,0.2005,...,20.3092,36.0244,0.2220,15.6043,24.2710,0.0883,9.9279,45.4026,0.4007,24.0964
1,cgm_time,CGM+Time,10.7638,0.0901,1.7825,7.3981,0.1048,1.4400,11.9251,0.1909,...,19.8888,35.8368,0.3005,15.6139,23.9976,0.1644,9.9145,45.2636,0.5006,24.1685
2,resnet18,+ ResNet18,10.7374,0.0788,1.7674,7.3781,0.0947,1.4237,11.8865,0.1742,...,19.7276,35.6261,0.2862,15.5409,23.7600,0.1777,9.8589,45.0259,0.4494,24.0621
3,vit,+ ViT-B/16,10.7408,0.0800,1.7766,7.3831,0.0966,1.4317,11.8960,0.1705,...,19.8047,35.7109,0.2718,15.5912,23.8666,0.1522,9.8972,45.1285,0.4562,24.1290
4,clip,+ CLIP ViT-B/16,10.7514,0.0773,1.7953,7.3808,0.0930,1.4467,11.9181,0.1672,...,19.8389,35.7084,0.2951,15.6148,23.8107,0.1814,9.9258,45.1486,0.4575,24.1765
5,dinov2,+ DINOv2 ViT-B/14,10.7386,0.0780,1.7489,7.3830,0.0914,1.4205,11.8767,0.1707,...,19.3631,35.8126,0.2551,15.2191,24.0831,0.1615,9.6827,45.1004,0.4068,23.6671


In [16]:
ceg_rows = []

for model, seed_dict in all_results.items():
    row = {'model_key': model, 'model': MODEL_NAMES[model]}

    for h in HORIZONS:
        zone_a = np.asarray([
            float(h_entry(seed_dict[seed]['test_results'], h)['ceg']['A']) for seed in SEEDS])
        zone_ab = np.asarray([
            float(h_entry(seed_dict[seed]['test_results'], h)['ceg']['A']) +
            float(h_entry(seed_dict[seed]['test_results'], h)['ceg']['B'])
            for seed in SEEDS
        ])

        row[f'ceg_a_{h}'] = float(np.mean(zone_a))
        row[f'ceg_a_sd1_{h}'] = float(np.std(zone_a, ddof=1))
        row[f'ceg_ab_{h}'] = float(np.mean(zone_ab))
        row[f'ceg_ab_sd1_{h}'] = float(np.std(zone_ab, ddof=1))

    ceg_rows.append(row)

ceg_summary = pd.DataFrame(ceg_rows)
display(ceg_summary.round(4))

,model_key,model,ceg_a_15,ceg_a_sd1_15,ceg_ab_15,ceg_ab_sd1_15,ceg_a_30,ceg_a_sd1_30,ceg_ab_30,ceg_ab_sd1_30,...,ceg_ab_60,ceg_ab_sd1_60,ceg_a_90,ceg_a_sd1_90,ceg_ab_90,ceg_ab_sd1_90,ceg_a_120,ceg_a_sd1_120,ceg_ab_120,ceg_ab_sd1_120
0,cgm,CGM,97.0484,0.1238,99.7746,0.0289,90.4424,0.1719,99.5178,0.0288,...,98.9363,0.0189,75.1249,0.2244,98.3214,0.0799,71.2504,0.3043,97.9861,0.1108
1,cgm_time,CGM+Time,97.4718,0.0959,99.7824,0.0120,91.7505,0.1720,99.5701,0.0159,...,99.0956,0.0437,75.8110,0.3979,98.3696,0.0746,71.8849,0.3543,97.8523,0.1322
2,resnet18,+ ResNet18,97.4878,0.0591,99.7830,0.0149,91.8599,0.1271,99.5642,0.0093,...,99.1515,0.0410,76.0352,0.4127,98.4576,0.0699,72.1370,0.3737,97.9213,0.1172
3,vit,+ ViT-B/16,97.4908,0.0589,99.7812,0.0137,91.7987,0.1451,99.5683,0.0120,...,99.1349,0.0462,75.8871,0.3448,98.4088,0.0807,72.0752,0.3250,97.8701,0.1212
4,clip,+ CLIP ViT-B/16,97.4836,0.0596,99.7788,0.0145,91.8962,0.1179,99.5659,0.0084,...,99.1652,0.0328,75.9407,0.4428,98.4600,0.0833,72.0032,0.3615,97.8725,0.1297
5,dinov2,+ DINOv2 ViT-B/14,97.4795,0.0495,99.7824,0.0149,91.8421,0.1302,99.5695,0.0138,...,99.1610,0.0510,75.5506,0.3971,98.4683,0.0920,71.4264,0.3692,97.9272,0.1024


In [17]:
improvement_rows = []

for h in HORIZONS:
    cgm = np.asarray([
        float(h_entry(all_results['cgm'][seed]['test_results'], h)['rmse']) for seed in SEEDS])
    cgm_time = np.asarray([
        float(h_entry(all_results['cgm_time'][seed]['test_results'], h)['rmse']) for seed in SEEDS])

    time_improvement = (cgm - cgm_time) / cgm * 100.0
    improvement_rows.append({
        'comparison': 'CGM -> CGM+Time',
        'horizon': int(h),
        'mean_improvement_pct': float(np.mean(time_improvement)),
        'sd_across_seeds_pct': float(np.std(time_improvement, ddof=1)),
        'improved_seeds': int(np.sum(cgm_time < cgm)),
    })

    for encoder in ENCODERS:
        image = np.asarray([
            float(h_entry(all_results[encoder][seed]['test_results'], h)['rmse']) for seed in SEEDS])
        image_improvement = (cgm_time - image) / cgm_time * 100.0

        improvement_rows.append({
            'comparison': f'CGM+Time -> {MODEL_NAMES[encoder]}',
            'horizon': int(h),
            'mean_improvement_pct': float(np.mean(image_improvement)),
            'sd_across_seeds_pct': float(np.std(image_improvement, ddof=1)),
            'improved_seeds': int(np.sum(image < cgm_time)),
        })

improvement_df = pd.DataFrame(improvement_rows)
display(improvement_df.round(4))

,comparison,horizon,mean_improvement_pct,sd_across_seeds_pct,improved_seeds
0,CGM -> CGM+Time,15,4.0820,1.3869,10
1,CGM+Time -> + ResNet18,15,0.2451,0.1763,9
2,CGM+Time -> + ViT-B/16,15,0.2129,0.2116,8
3,CGM+Time -> + CLIP ViT-B/16,15,0.1143,0.2092,7
4,CGM+Time -> + DINOv2 ViT-B/14,15,0.2338,0.2080,9
5,CGM -> CGM+Time,30,6.7914,0.7889,10
6,CGM+Time -> + ResNet18,30,0.4848,0.1908,10
7,CGM+Time -> + ViT-B/16,30,0.3932,0.2922,10
8,CGM+Time -> + CLIP ViT-B/16,30,0.3054,0.1949,9
9,CGM+Time -> + DINOv2 ViT-B/14,30,0.4439,0.2689,10


In [18]:
def make_numeric_table(summary, metric):
    table = pd.DataFrame({'Model': summary['model']})

    for h in HORIZONS:
        def fmt(row):
            mean = row[f'{metric}_{h}']
            sd1 = row[f'{metric}_sd1_{h}']
            sd2 = row[f'{metric}_sd2_{h}']
            return f'{mean:.2f} ± {sd1:.2f} ({sd2:.2f})' if pd.notna(sd2) else f'{mean:.2f} ± {sd1:.2f}'
        table[f'{h} min'] = summary.apply(fmt, axis=1)

    return table


rmse_table = make_numeric_table(numeric_summary, 'rmse')
mae_table = make_numeric_table(numeric_summary, 'mae')
grmse_table = make_numeric_table(numeric_summary, 'grmse')

print('RMSE: mean ± sd1 (sd2)')
display(rmse_table)

print('MAE: mean ± sd1 (sd2)')
display(mae_table)

print('gRMSE: mean ± sd1 (sd2)')
display(grmse_table)

RMSE: mean ± sd1 (sd2)


,Model,15 min,30 min,60 min,90 min,120 min
0,CGM,11.22 ± 0.09 (1.89),18.55 ± 0.08 (3.85),27.50 ± 0.17 (9.15),32.96 ± 0.20 (13.10),36.02 ± 0.22 (15.60)
1,CGM+Time,10.76 ± 0.09 (1.78),17.29 ± 0.09 (3.34),26.26 ± 0.20 (8.30),32.36 ± 0.28 (12.71),35.84 ± 0.30 (15.61)
2,+ ResNet18,10.74 ± 0.08 (1.77),17.20 ± 0.08 (3.26),26.08 ± 0.19 (8.18),32.16 ± 0.26 (12.58),35.63 ± 0.29 (15.54)
3,+ ViT-B/16,10.74 ± 0.08 (1.78),17.22 ± 0.09 (3.27),26.13 ± 0.18 (8.21),32.24 ± 0.25 (12.64),35.71 ± 0.27 (15.59)
4,+ CLIP ViT-B/16,10.75 ± 0.08 (1.80),17.23 ± 0.08 (3.30),26.12 ± 0.20 (8.23),32.23 ± 0.28 (12.66),35.71 ± 0.30 (15.61)
5,+ DINOv2 ViT-B/14,10.74 ± 0.08 (1.75),17.21 ± 0.10 (3.20),26.14 ± 0.19 (7.98),32.29 ± 0.24 (12.30),35.81 ± 0.26 (15.22)


MAE: mean ± sd1 (sd2)


,Model,15 min,30 min,60 min,90 min,120 min
0,CGM,7.60 ± 0.09 (1.46),12.32 ± 0.09 (2.62),18.20 ± 0.10 (5.74),21.98 ± 0.09 (8.09),24.27 ± 0.09 (9.93)
1,CGM+Time,7.40 ± 0.10 (1.44),11.66 ± 0.08 (2.45),17.40 ± 0.10 (5.32),21.50 ± 0.16 (7.92),24.00 ± 0.16 (9.91)
2,+ ResNet18,7.38 ± 0.09 (1.42),11.61 ± 0.07 (2.38),17.27 ± 0.11 (5.22),21.31 ± 0.17 (7.81),23.76 ± 0.18 (9.86)
3,+ ViT-B/16,7.38 ± 0.10 (1.43),11.63 ± 0.08 (2.39),17.32 ± 0.09 (5.25),21.40 ± 0.15 (7.85),23.87 ± 0.15 (9.90)
4,+ CLIP ViT-B/16,7.38 ± 0.09 (1.45),11.63 ± 0.07 (2.42),17.30 ± 0.11 (5.28),21.35 ± 0.17 (7.89),23.81 ± 0.18 (9.93)
5,+ DINOv2 ViT-B/14,7.38 ± 0.09 (1.42),11.62 ± 0.08 (2.39),17.35 ± 0.11 (5.17),21.53 ± 0.16 (7.68),24.08 ± 0.16 (9.68)


gRMSE: mean ± sd1 (sd2)


,Model,15 min,30 min,60 min,90 min,120 min
0,CGM,12.42 ± 0.20 (2.99),21.50 ± 0.20 (6.38),33.60 ± 0.35 (14.34),41.14 ± 0.37 (20.31),45.40 ± 0.40 (24.10)
1,CGM+Time,11.93 ± 0.19 (2.88),19.94 ± 0.22 (5.79),31.94 ± 0.36 (13.40),40.38 ± 0.45 (19.89),45.26 ± 0.50 (24.17)
2,+ ResNet18,11.89 ± 0.17 (2.85),19.81 ± 0.19 (5.67),31.68 ± 0.31 (13.23),40.12 ± 0.39 (19.73),45.03 ± 0.45 (24.06)
3,+ ViT-B/16,11.90 ± 0.17 (2.86),19.84 ± 0.20 (5.70),31.76 ± 0.33 (13.29),40.23 ± 0.42 (19.80),45.13 ± 0.46 (24.13)
4,+ CLIP ViT-B/16,11.92 ± 0.17 (2.89),19.88 ± 0.19 (5.75),31.76 ± 0.31 (13.33),40.23 ± 0.42 (19.84),45.15 ± 0.46 (24.18)
5,+ DINOv2 ViT-B/14,11.88 ± 0.17 (2.81),19.80 ± 0.19 (5.57),31.67 ± 0.30 (12.96),40.15 ± 0.37 (19.36),45.10 ± 0.41 (23.67)


In [19]:
ceg_a_table = pd.DataFrame({'Model': ceg_summary['model']})
ceg_ab_table = pd.DataFrame({'Model': ceg_summary['model']})

for h in HORIZONS:
    ceg_a_table[f'{h} min'] = ceg_summary.apply(
        lambda r: f'{r[f"ceg_a_{h}"]:.2f} ± {r[f"ceg_a_sd1_{h}"]:.2f}', axis=1)
    ceg_ab_table[f'{h} min'] = ceg_summary.apply(
        lambda r: f'{r[f"ceg_ab_{h}"]:.2f} ± {r[f"ceg_ab_sd1_{h}"]:.2f}', axis=1)

print('Clarke Error Grid Zone A (%): mean ± sd1')
display(ceg_a_table)

print('Clarke Error Grid Zone A+B (%): mean ± sd1')
display(ceg_ab_table)

Clarke Error Grid Zone A (%): mean ± sd1


,Model,15 min,30 min,60 min,90 min,120 min
0,CGM,97.05 ± 0.12,90.44 ± 0.17,81.36 ± 0.16,75.12 ± 0.22,71.25 ± 0.30
1,CGM+Time,97.47 ± 0.10,91.75 ± 0.17,82.32 ± 0.26,75.81 ± 0.40,71.88 ± 0.35
2,+ ResNet18,97.49 ± 0.06,91.86 ± 0.13,82.53 ± 0.22,76.04 ± 0.41,72.14 ± 0.37
3,+ ViT-B/16,97.49 ± 0.06,91.80 ± 0.15,82.42 ± 0.21,75.89 ± 0.34,72.08 ± 0.32
4,+ CLIP ViT-B/16,97.48 ± 0.06,91.90 ± 0.12,82.55 ± 0.19,75.94 ± 0.44,72.00 ± 0.36
5,+ DINOv2 ViT-B/14,97.48 ± 0.05,91.84 ± 0.13,82.48 ± 0.23,75.55 ± 0.40,71.43 ± 0.37


Clarke Error Grid Zone A+B (%): mean ± sd1


,Model,15 min,30 min,60 min,90 min,120 min
0,CGM,99.77 ± 0.03,99.52 ± 0.03,98.94 ± 0.02,98.32 ± 0.08,97.99 ± 0.11
1,CGM+Time,99.78 ± 0.01,99.57 ± 0.02,99.10 ± 0.04,98.37 ± 0.07,97.85 ± 0.13
2,+ ResNet18,99.78 ± 0.01,99.56 ± 0.01,99.15 ± 0.04,98.46 ± 0.07,97.92 ± 0.12
3,+ ViT-B/16,99.78 ± 0.01,99.57 ± 0.01,99.13 ± 0.05,98.41 ± 0.08,97.87 ± 0.12
4,+ CLIP ViT-B/16,99.78 ± 0.01,99.57 ± 0.01,99.17 ± 0.03,98.46 ± 0.08,97.87 ± 0.13
5,+ DINOv2 ViT-B/14,99.78 ± 0.01,99.57 ± 0.01,99.16 ± 0.05,98.47 ± 0.09,97.93 ± 0.10


In [20]:
numeric_summary.to_csv(SUMMARY_DIR / 'final_all_models_numeric_metrics.csv', index=False)
ceg_summary.to_csv(SUMMARY_DIR / 'final_all_models_ceg.csv', index=False)
improvement_df.to_csv(SUMMARY_DIR / 'final_incremental_improvements.csv', index=False)

rmse_table.to_csv(SUMMARY_DIR / 'final_rmse_table.csv', index=False)
mae_table.to_csv(SUMMARY_DIR / 'final_mae_table.csv', index=False)
grmse_table.to_csv(SUMMARY_DIR / 'final_grmse_table.csv', index=False)
ceg_a_table.to_csv(SUMMARY_DIR / 'final_ceg_zone_a_table.csv', index=False)
ceg_ab_table.to_csv(SUMMARY_DIR / 'final_ceg_zone_ab_table.csv', index=False)

print('Saved final RQ1 summaries to:')
print(SUMMARY_DIR)

Saved final RQ1 summaries to:
/content/drive/MyDrive/dissertation/experiments/6_final_multimodal/summary
